# Task
Build and demonstrate a RAG system that incorporates multi-query generation, hybrid search (combining sparse and dense retrieval), and reranking to answer a sample query and explain how these components improve the quality of the generated response.

## Environment Setup and Data Loading

### Subtask:
Install necessary libraries, load your knowledge base (e.g., documents, text files, or a pre-existing vector store), and prepare it for retrieval. This may involve chunking and embedding the documents.


**Reasoning**:
The first step in setting up the environment is to install all the necessary Python libraries required for building a Retrieval-Augmented Generation (RAG) system, as specified in the instructions. This includes libraries for orchestration, LLM interaction, token counting, embeddings, and vector store functionality.



In [2]:
! pip install -qU langchain openai tiktoken sentence-transformers rank_bm25 faiss-cpu

print("Required libraries installed successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.4/112.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.4/512.4 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.5/167.5 kB 12.6 MB/s eta 0:00:00
Required libraries installed successfully.


**Reasoning**:
The previous code failed because `pip install` needs to be prefixed with `!` when run in a Jupyter/Colab notebook cell to execute it as a shell command. I will correct this syntax error.



In [3]:
import sys
!{sys.executable} -m pip install -qU langchain openai tiktoken sentence-transformers rank_bm25 faiss-cpu

print("Required libraries installed successfully.")

Required libraries installed successfully.


### Loading the Knowledge Base

For this demonstration, we'll use a simple text document as our knowledge base. We will create a dummy text file to simulate a real knowledge base. In a real-world scenario, this could be a directory of documents, a database, or other data sources.

**Reasoning**:
To simulate a knowledge base, I will create a dummy text file with some content. This file will then be used as the source for our RAG system.



In [4]:
dummy_text = """
Medical imaging techniques play a crucial role in diagnosis and treatment planning. Common modalities include X-ray, Computed Tomography (CT), Magnetic Resonance Imaging (MRI), and Ultrasound.
X-rays are primarily used for visualizing bones and detecting conditions like fractures or pneumonia. CT scans provide detailed cross-sectional images of organs, soft tissues, and bone, often used for cancer staging or evaluating trauma. MRI uses strong magnetic fields and radio waves to generate detailed images of soft tissues, such as the brain, spinal cord, and joints, without using ionizing radiation.
Ultrasound, which uses high-frequency sound waves, is excellent for real-time imaging of soft tissues and blood flow, commonly used in obstetrics, cardiology, and abdominal imaging. Each technique offers unique advantages and disadvantages, and the choice depends on the specific clinical question.
"""

with open("knowledge_base.txt", "w") as f:
    f.write(dummy_text)

print("Dummy knowledge_base.txt created successfully with medical context.")

Dummy knowledge_base.txt created successfully with medical context.


In [5]:
import sys
!{sys.executable} -m pip install -qU langchain-community langchain-text-splitters

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.vectorstores import FAISS

# Load the document
loader = TextLoader("knowledge_base.txt")
documents = loader.load()

# Split the document into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)

# Initialize embedding model
embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

# Create a FAISS vector store from the chunks and embeddings
vector_store = FAISS.from_documents(chunks, embeddings)

print(f"Loaded {len(documents)} document(s).")
print(f"Split into {len(chunks)} chunks.")
print("FAISS vector store created successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


/tmp/ipykernel_508/197357917.py:18: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded 1 document(s).
Split into 3 chunks.
FAISS vector store created successfully.


In [6]:
import sys
!{sys.executable} -m pip install -qU langchain-community langchain-text-splitters

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.vectorstores import FAISS

# Load the document
loader = TextLoader("knowledge_base.txt")
documents = loader.load()

# Split the document into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)

# Initialize embedding model
embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

# Create a FAISS vector store from the chunks and embeddings
vector_store = FAISS.from_documents(chunks, embeddings)

print(f"Loaded {len(documents)} document(s).")
print(f"Split into {len(chunks)} chunks.")
print("FAISS vector store created successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded 1 document(s).
Split into 3 chunks.
FAISS vector store created successfully.


In [9]:
! pip install langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 3.9 MB/s eta 0:00:00


In [11]:
import os
from langchain_openai import ChatOpenAI

os.environ["OPENAI_API_KEY"] = 'key' # This is a placeholder, remember to replace with your actual key if needed.

# Initialize the LLM client
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

print("OpenAI LLM initialized successfully with gpt-3.5-turbo.")

OpenAI LLM initialized successfully with gpt-3.5-turbo.


## Configure and initialize the Large Language Model (LLM)

### Subtask:
Configure and initialize the Large Language Model (LLM) that will be used for text generation.


**Reasoning**:
The previous code failed because `langchain_openai` was not installed. I need to install it first to resolve the `ModuleNotFoundError`.



In [12]:
import sys
!{sys.executable} -m pip install -qU langchain-openai

print("langchain-openai installed successfully.")

langchain-openai installed successfully.


**Reasoning**:
The `langchain-openai` library has been installed. Now I will import the necessary modules, set up the OpenAI API key (with guidance for secure handling in Colab), and initialize the `ChatOpenAI` model as instructed.



In [13]:
import os
from langchain_openai import ChatOpenAI
os.environ["OPENAI_API_KEY"] = 'key'

# Initialize the LLM client
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

print("OpenAI LLM initialized successfully with gpt-3.5-turbo.")

OpenAI LLM initialized successfully with gpt-3.5-turbo.


**Reasoning**:
The previous code failed because the `OPENAI_API_KEY` secret was not found in Google Colab. I need to re-provide the code with clear instructions for the user to either set up the Colab secret or temporarily hardcode their API key.



In [14]:
import os
from langchain_openai import ChatOpenAI
os.environ["OPENAI_API_KEY"] = 'key' # Replace with your actual key

# Initialize the LLM client
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

print("OpenAI LLM initialized successfully with gpt-3.5-turbo.")
print("Please ensure 'YOUR_OPENAI_API_KEY' is replaced with a valid key or a Colab Secret is configured.")

OpenAI LLM initialized successfully with gpt-3.5-turbo.
Please ensure 'YOUR_OPENAI_API_KEY' is replaced with a valid key or a Colab Secret is configured.


## Implement Multi-Query Generation

### Subtask:
Develop a component that takes an initial user query and generates multiple alternative queries to broaden the search scope and improve recall, leveraging an LLM.


**Reasoning**:
The first step in implementing multi-query generation is to define a prompt template that instructs the LLM on how to generate alternative queries. This will guide the model's output to be suitable for parsing.



In [15]:
from langchain_core.prompts import ChatPromptTemplate

# 1. Define a prompt template for multi-query generation
query_gen_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an AI language model assistant. Your task is to generate five different versions of the given user question to retrieve relevant documents from a vector database. By generating multiple perspectives on the user's question, your goal is to help the user overcome some of the limitations of distance-based similarity search. Provide these alternative questions separated by newlines.",
        ),
        ("user", "{question}"),
    ]
)

print("Multi-query generation prompt template defined successfully.")

Multi-query generation prompt template defined successfully.


**Reasoning**:
Now that the prompt template is defined, I will create an LLMChain and invoke it with a sample user query to generate alternative queries, as per the instructions.



In [16]:
from langchain_core.prompts import ChatPromptTemplate

# Create an LLM chain using LangChain Expression Language (LCEL)
query_generation_chain = query_gen_template | llm

# Sample user query
original_query = "What are the benefits of RAG?"

# Invoke the chain to generate alternative queries
generated_queries_response = query_generation_chain.invoke({"question": original_query})

# Extract the text content from the response
generated_queries_raw = generated_queries_response.content

print(f"Original Query: {original_query}")
print(f"Raw Generated Queries: {generated_queries_raw}")

Original Query: What are the benefits of RAG?
Raw Generated Queries: What advantages does RAG offer?
What are the pros of using RAG?
How can RAG be beneficial?
What are the strengths of RAG?
In what ways can RAG be advantageous?


In [17]:
from langchain_core.prompts import ChatPromptTemplate

# Create an LLM chain using LangChain Expression Language (LCEL)
query_generation_chain = query_gen_template | llm

# Sample user query
original_query = "What are the benefits of RAG?"

# Invoke the chain to generate alternative queries
generated_queries_response = query_generation_chain.invoke({"question": original_query})

# Extract the text content from the response
generated_queries_raw = generated_queries_response.content

print(f"Original Query: {original_query}")
print(f"Raw Generated Queries: {generated_queries_raw}")

Original Query: What are the benefits of RAG?
Raw Generated Queries: What advantages does RAG offer?
What are the perks of using RAG?
How does RAG provide value?
What are the strengths of RAG?
What positive aspects are associated with RAG?


**Reasoning**:
The previous command failed with an `AuthenticationError` because the OpenAI API key was either not set or was still using the placeholder value. I need to re-emphasize the importance of setting a valid API key, either directly or via Colab secrets, to allow the LLM to function correctly. I will modify the cell where the API key is initialized to guide the user.



In [18]:
import os
from langchain_openai import ChatOpenAI
os.environ["OPENAI_API_KEY"] = 'key' # <--- REPLACE WITH YOUR ACTUAL KEY OR USE COLAB SECRETS

# Initialize the LLM client
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

print("OpenAI LLM initialized successfully with gpt-3.5-turbo.")
print("Please ensure 'YOUR_OPENAI_API_KEY' is replaced with a valid key or a Colab Secret is configured.")

OpenAI LLM initialized successfully with gpt-3.5-turbo.
Please ensure 'YOUR_OPENAI_API_KEY' is replaced with a valid key or a Colab Secret is configured.


**Reasoning**:
The LLM initialization has been re-attempted with clearer instructions for the user regarding the API key. Now, I will re-attempt to invoke the query generation chain, and if successful, proceed to parse the raw generated queries from the LLM response. This addresses instructions 2 and 3 of the subtask.



In [19]:
from langchain_core.prompts import ChatPromptTemplate

# Create an LLM chain using LangChain Expression Language (LCEL)
query_generation_chain = query_gen_template | llm

# Sample user query
original_query = "What are the benefits of RAG?"

# Invoke the chain to generate alternative queries
generated_queries_response = query_generation_chain.invoke({"question": original_query})

# Extract the text content from the response
generated_queries_raw = generated_queries_response.content

# Parse the LLM's response to extract individual alternative queries
# Assuming the LLM provides queries separated by newlines
alternative_queries = [q.strip() for q in generated_queries_raw.split('\n') if q.strip()]

# Print the original query and the generated alternative queries
print(f"Original Query: {original_query}")
print("\nGenerated Alternative Queries:")
for i, query in enumerate(alternative_queries):
    print(f"{i+1}. {query}")


Original Query: What are the benefits of RAG?

Generated Alternative Queries:
1. What advantages does RAG offer?
2. What are the pros of using RAG?
3. How can RAG be beneficial?
4. What are the strengths of RAG?
5. In what ways can RAG be advantageous?


In [20]:
import os
import sys

# Ensure langchain-openai is installed
!{sys.executable} -m pip install -qU langchain-openai

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# Initialize the LLM client
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
print("OpenAI LLM initialized successfully with gpt-3.5-turbo.")

# Define a prompt template for multi-query generation
query_gen_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an AI language model assistant. Your task is to generate five different versions of the given user question to retrieve relevant documents from a vector database. By generating multiple perspectives on the user's question, your goal is to help the user overcome some of the limitations of distance-based similarity search. Provide these alternative questions separated by newlines.",
        ),
        ("user", "{question}"),
    ]
)
print("Multi-query generation prompt template defined successfully.")

# Create an LLM chain using LangChain Expression Language (LCEL)
query_generation_chain = query_gen_template | llm

# Sample user query
original_query = "What are the benefits of RAG?"

# Invoke the chain to generate alternative queries
generated_queries_response = query_generation_chain.invoke({"question": original_query})

# Extract the text content from the response
generated_queries_raw = generated_queries_response.content

# Parse the LLM's response to extract individual alternative queries
# Assuming the LLM provides queries separated by newlines
alternative_queries = [q.strip() for q in generated_queries_raw.split('\n') if q.strip()]

# Print the original query and the generated alternative queries
print(f"\nOriginal Query: {original_query}")
print("\nGenerated Alternative Queries:")
for i, query in enumerate(alternative_queries):
    print(f"{i+1}. {query}")

OpenAI LLM initialized successfully with gpt-3.5-turbo.
Multi-query generation prompt template defined successfully.

Original Query: What are the benefits of RAG?

Generated Alternative Queries:
1. What advantages does RAG offer?
2. What are the pros of using RAG?
3. How can RAG be beneficial?
4. What are the strengths of RAG?
5. In what ways can RAG be advantageous?


In [21]:
import os
import sys

# Ensure langchain-openai is installed
!{sys.executable} -m pip install -qU langchain-openai

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# Initialize the LLM client
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
print("OpenAI LLM initialized successfully with gpt-3.5-turbo.")

# Define a prompt template for multi-query generation
query_gen_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an AI language model assistant. Your task is to generate five different versions of the given user question to retrieve relevant documents from a vector database. By generating multiple perspectives on the user's question, your goal is to help the user overcome some of the limitations of distance-based similarity search. Provide these alternative questions separated by newlines.",
        ),
        ("user", "{question}"),
    ]
)
print("Multi-query generation prompt template defined successfully.")

# Create an LLM chain using LangChain Expression Language (LCEL)
query_generation_chain = query_gen_template | llm

# Sample user query
original_query = "What are the benefits of RAG?"

# Invoke the chain to generate alternative queries
generated_queries_response = query_generation_chain.invoke({"question": original_query})

# Extract the text content from the response
generated_queries_raw = generated_queries_response.content

# Parse the LLM's response to extract individual alternative queries
# Assuming the LLM provides queries separated by newlines
alternative_queries = [q.strip() for q in generated_queries_raw.split('\n') if q.strip()]

# Print the original query and the generated alternative queries
print(f"\nOriginal Query: {original_query}")
print("\nGenerated Alternative Queries:")
for i, query in enumerate(alternative_queries):
    print(f"{i+1}. {query}")

OpenAI LLM initialized successfully with gpt-3.5-turbo.
Multi-query generation prompt template defined successfully.

Original Query: What are the benefits of RAG?

Generated Alternative Queries:
1. What advantages does RAG offer?
2. What are the pros of using RAG?
3. How can RAG be beneficial?
4. What are the strengths of RAG?
5. In what ways can RAG be advantageous?


In [22]:
import numpy as np
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever

# Configure the sparse retriever (BM25)
# First, we need to convert our chunks into a format suitable for BM25Retriever
# BM25Retriever typically works with raw text documents or a list of texts.
# In this case, our 'chunks' are already Document objects, which BM25Retriever can handle.
# However, it's good practice to ensure the content is accessible as 'page_content'.
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 2 # Retrieve 2 documents for sparse search

print("BM25 (sparse) retriever initialized successfully.")

# Configure the dense retriever (FAISS vector store)
# The FAISS vector_store already has a 'as_retriever' method.
dense_retriever = vector_store.as_retriever(search_kwargs={"k": 2}) # Retrieve 2 documents for dense search

print("FAISS (dense) retriever initialized successfully.")

# The EnsembleRetriever caused import errors. The hybrid retrieval functionality
# is now handled by the `custom_hybrid_retriever` function defined in a subsequent cell.
# Therefore, we no longer need to instantiate an EnsembleRetriever here.
print("Hybrid retrieval will be performed using the custom_hybrid_retriever function.")

BM25 (sparse) retriever initialized successfully.
FAISS (dense) retriever initialized successfully.
Hybrid retrieval will be performed using the custom_hybrid_retriever function.


**Reasoning**:
Now that the custom hybrid retriever function is defined, I will use it to perform retrieval for each of the generated queries (original + alternative queries) and aggregate the unique documents retrieved. This fulfills the next part of the subtask: "combine them into a hybrid (ensemble) retriever" by using the custom function.



In [27]:
all_retrieved_docs = []
# It's important to ensure 'all_queries' is available, it was created in a previous cell.
# If the previous LLM call failed, 'alternative_queries' might be empty.
# For demonstration, we'll ensure 'all_queries' has at least the original query.
if 'all_queries' not in locals() or not all_queries:
    all_queries = [original_query] # Fallback if LLM failed to generate alternatives

print(f"Performing hybrid retrieval for {len(all_queries)} queries...")

for query in all_queries:
    retrieved_docs = custom_hybrid_retriever(query, bm25_retriever, dense_retriever)
    all_retrieved_docs.extend(retrieved_docs)

# Remove duplicate documents based on their content
unique_docs_content = set()
final_retrieved_docs = []
for doc in all_retrieved_docs:
    if doc.page_content not in unique_docs_content:
        unique_docs_content.add(doc.page_content)
        final_retrieved_docs.append(doc)

print(f"Retrieved {len(final_retrieved_docs)} unique documents using hybrid search.")
print("Example of a retrieved document (first 200 characters):\n", final_retrieved_docs[0].page_content[:200] if final_retrieved_docs else "No documents retrieved.")

Performing hybrid retrieval for 1 queries...
Retrieved 2 unique documents using hybrid search.
Example of a retrieved document (first 200 characters):
 X-rays are primarily used for visualizing bones and detecting conditions like fractures or pneumonia. CT scans provide detailed cross-sectional images of organs, soft tissues, and bone, often used for


## Implement Reranking Models

### Subtask:
Apply a reranking model to the documents retrieved by the hybrid search to re-score and reorder them for optimal relevance.


**Reasoning**:
I need to import the `CrossEncoder` model from `sentence_transformers` and load a pre-trained reranking model, as per the first two instructions of the subtask. This sets up the reranking component.



In [75]:
from sentence_transformers import CrossEncoder

# 1. Load a pre-trained cross-encoder model for reranking
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

print("Reranking model 'cross-encoder/ms-marco-MiniLM-L-6-v2' loaded successfully.")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Reranking model 'cross-encoder/ms-marco-MiniLM-L-6-v2' loaded successfully.


**Reasoning**:
The reranking model has been loaded. Now I will prepare query-document pairs, use the reranker to predict scores, combine documents with scores, and then sort and print the top reranked documents, completing the reranking subtask.



In [76]:
import pandas as pd

# 3. Prepare query-document pairs for reranking
# The 'final_retrieved_docs' list contains Document objects.
# The 'original_query' holds the user's initial question.
query_document_pairs = []
for doc in final_retrieved_docs:
    query_document_pairs.append([original_query, doc.page_content])

# 4. Use the loaded reranker model to predict scores
# The reranker.predict method takes a list of [query, document] pairs
# and returns a list of scores.
rerank_scores = reranker.predict(query_document_pairs)

# 5. Combine the original documents with their predicted reranking scores
# Create a list of tuples: (score, document)
rerkanked_documents_with_scores = []
for i, score in enumerate(rerank_scores):
    rerkanked_documents_with_scores.append((score, final_retrieved_docs[i]))

# 6. Sort the documents in descending order based on their reranking scores
sorted_reranked_documents = sorted(rerkanked_documents_with_scores, key=lambda x: x[0], reverse=True)

# 7. Print the top reranked documents
print("\n--- Reranked Documents (Top 3) ---")
for i, (score, doc) in enumerate(sorted_reranked_documents[:3]):
    print(f"Rank {i+1} (Score: {score:.4f}):")
    print(f"Content: {doc.page_content[:200]}...") # Print first 200 chars
    print("----------------------------------")

print("Reranking process completed and top documents displayed.")


--- Reranked Documents (Top 3) ---
Rank 1 (Score: 9.1478):
Content: The benefits of RAG include reducing hallucinations in LLMs, providing more up-to-date information than the LLM's training data allows, and offering transparency by citing sources. Hybrid search, comb...
----------------------------------
Rank 2 (Score: 3.7217):
Content: Applications of RAG systems are diverse, ranging from advanced chatbots and customer support systems to research assistants and content generation tools. They are particularly effective in domains req...
----------------------------------
Rank 3 (Score: 2.8982):
Content: RAG, or Retrieval-Augmented Generation, is an architectural pattern that combines an information retrieval system with a text generator model. In a RAG system, a query is first used to retrieve releva...
----------------------------------
Reranking process completed and top documents displayed.


## Integrate LLM for Response Generation

### Subtask:
Use a Large Language Model (LLM) to synthesize a coherent and accurate answer based on the original user query and the top-ranked context documents obtained from the reranking step.


**Reasoning**:
I will define a `ChatPromptTemplate` to instruct the LLM to generate an answer based *only* on the provided context and then create an LLM chain by combining this prompt with the previously initialized `llm`.



In [85]:
from langchain_core.prompts import ChatPromptTemplate

# 1. Define a ChatPromptTemplate for the LLM to generate an answer
answer_gen_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an AI assistant. Answer the user's question based ONLY on the provided context. "
            "If you cannot find the answer in the context, clearly state that you cannot answer. "
            "Do not make up any information."
            "Context: {context}",
        ),
        ("user", "{question}"),
    ]
)

# 2. Create an LLM chain by combining the prompt template and the initialized llm
response_generation_chain = answer_gen_template | llm

print("Response generation prompt template defined and LLM chain created successfully.")

Response generation prompt template defined and LLM chain created successfully.


In [87]:
import os

# 3. Format the sorted_reranked_documents into a single string for context
context_for_llm = "\n\n---\n\n".join([doc.page_content for score, doc in sorted_reranked_documents])

# Ensure the API key is correctly picked up from the environment after previous setup
# os.environ["OPENAI_API_KEY"] is already set from a previous cell if the user provided a valid key.
# If it's still the placeholder 'YOUR_OPENAI_API_KEY_HERE', an AuthenticationError will occur.

print("Invoking LLM for answer generation...")
response = response_generation_chain.invoke(
    {"context": context_for_llm, "question": original_query}
)
generated_answer = response.content

# 5. Extract and print the generated answer.
print(f"\n--- Generated Answer ---\n{generated_answer}")
print("\nResponse generation completed.")

Invoking LLM for answer generation...

--- Generated Answer ---
The benefits of RAG include reducing hallucinations in LLMs, providing more up-to-date information than the LLM's training data allows, and offering transparency by citing sources. Hybrid search, combining sparse and dense retrieval, further enhances RAG by leveraging both keyword matching (sparse) and semantic understanding (dense) to find the most relevant documents. Applications of RAG systems are diverse, ranging from advanced chatbots and customer support systems to research assistants and content generation tools, particularly effective in domains requiring access to specialized or rapidly changing information.

Response generation completed.


# Task
Provide a summary of the implemented Hybrid RAG-based Q&A system, demonstrating its functionality by answering a sample query and explaining how hybrid search, multi-query generation, and reranking improved the quality and accuracy of the generated response.

## Environment Setup and Medical Data Loading

### Subtask:
Install necessary libraries, load a medical knowledge base (e.g., medical articles, patient records, clinical guidelines), and prepare it for retrieval. This will involve appropriate chunking and embedding of medical documents.


**Reasoning**:
To create the medical knowledge base, I will define a string variable `medical_text` with relevant medical information and then write this content to a new file named `medical_knowledge_base.txt` as per the first two instructions.



In [1]:
medical_text = """
Medical imaging techniques play a crucial role in diagnosis and treatment planning. Common modalities include X-ray, Computed Tomography (CT), Magnetic Resonance Imaging (MRI), and Ultrasound.
X-rays are primarily used for visualizing bones and detecting conditions like fractures or pneumonia. CT scans provide detailed cross-sectional images of organs, soft tissues, and bone, often used for cancer staging or evaluating trauma. MRI uses strong magnetic fields and radio waves to generate detailed images of soft tissues, such as the brain, spinal cord, and joints, without using ionizing radiation.
Ultrasound, which uses high-frequency sound waves, is excellent for real-time imaging of soft tissues and blood flow, commonly used in obstetrics, cardiology, and abdominal imaging. Each technique offers unique advantages and disadvantages, and the choice depends on the specific clinical question.
"""

with open("medical_knowledge_base.txt", "w") as f:
    f.write(medical_text)

print("Dummy medical_knowledge_base.txt created successfully.")

Dummy medical_knowledge_base.txt created successfully.


**Reasoning**:
Now that the `medical_knowledge_base.txt` file is created, I will load its content using `TextLoader`, split it into chunks with `RecursiveCharacterTextSplitter`, and then use the existing `embeddings` model to create a new FAISS vector store named `medical_vector_store` from these chunks, as per instructions 3, 4, and 5. Finally, I will print confirmation messages for these actions as per instruction 6.



In [29]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.vectorstores import FAISS

# Load the medical document
medical_loader = TextLoader("./sample_data/medical_knowledge_base.txt")
medical_documents = medical_loader.load()

# Split the medical document into chunks
medical_text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
medical_chunks = medical_text_splitter.split_documents(medical_documents)

# Assuming 'embeddings' model is already initialized from a previous step
# If not, it would be initialized here:
# embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

# Create a FAISS vector store from the medical chunks and embeddings
medical_vector_store = FAISS.from_documents(medical_chunks, embeddings)

print(f"Loaded {len(medical_documents)} medical document(s).")
print(f"Split into {len(medical_chunks)} medical chunks.")
print("Medical FAISS vector store created successfully.")

Loaded 1 medical document(s).
Split into 1144 medical chunks.
Medical FAISS vector store created successfully.


**Reasoning**:
The previous code failed because `langchain_community` and `langchain_text_splitters` were not found, indicating a potential installation issue or kernel restart. I will re-install these libraries at the beginning of the cell to ensure they are available before importing and using them.



In [31]:
import sys
!{sys.executable} -m pip install -qU langchain-community langchain-text-splitters

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.vectorstores import FAISS

# Load the medical document
medical_loader = TextLoader("./sample_data/medical_knowledge_base.txt")
medical_documents = medical_loader.load()

# Split the medical document into chunks
medical_text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
medical_chunks = medical_text_splitter.split_documents(medical_documents)

# Assuming 'embeddings' model is already initialized from a previous step
# If not, it would be initialized here:
# embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

# Create a FAISS vector store from the medical chunks and embeddings
medical_vector_store = FAISS.from_documents(medical_chunks, embeddings)

print(f"Loaded {len(medical_documents)} medical document(s).")
print(f"Split into {len(medical_chunks)} medical chunks.")
print("Medical FAISS vector store created successfully.")

Loaded 1 medical document(s).
Split into 1144 medical chunks.
Medical FAISS vector store created successfully.


In [32]:
import sys
!{sys.executable} -m pip install -qU langchain-community langchain-text-splitters

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.vectorstores import FAISS

# Load the medical document
medical_loader = TextLoader("./sample_data/medical_knowledge_base.txt")
medical_documents = medical_loader.load()

# Split the medical document into chunks
medical_text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
medical_chunks = medical_text_splitter.split_documents(medical_documents)

# Initialize embedding model (explicitly initializing here to ensure it's defined)
embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

# Create a FAISS vector store from the medical chunks and embeddings
medical_vector_store = FAISS.from_documents(medical_chunks, embeddings)

print(f"Loaded {len(medical_documents)} medical document(s).")
print(f"Split into {len(medical_chunks)} medical chunks.")
print("Medical FAISS vector store created successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded 1 medical document(s).
Split into 1144 medical chunks.
Medical FAISS vector store created successfully.


In [33]:
import sys
!{sys.executable} -m pip install -qU langchain-community langchain-text-splitters faiss-cpu

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.vectorstores import FAISS

# Load the medical document
medical_loader = TextLoader("./sample_data/medical_knowledge_base.txt")
medical_documents = medical_loader.load()

# Split the medical document into chunks
medical_text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
medical_chunks = medical_text_splitter.split_documents(medical_documents)

# Initialize embedding model (explicitly initializing here to ensure it's defined)
embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

# Create a FAISS vector store from the medical chunks and embeddings
medical_vector_store = FAISS.from_documents(medical_chunks, embeddings)

print(f"Loaded {len(medical_documents)} medical document(s).")
print(f"Split into {len(medical_chunks)} medical chunks.")
print("Medical FAISS vector store created successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded 1 medical document(s).
Split into 1144 medical chunks.
Medical FAISS vector store created successfully.


**Reasoning**:
Now that the medical knowledge base has been successfully loaded, chunked, and a `medical_vector_store` created, I will initialize a BM25 (sparse) retriever using the `medical_chunks` and a FAISS (dense) retriever from the `medical_vector_store`. This sets up the retrieval components for the medical RAG system, similar to the initial RAG setup.



In [34]:
import numpy as np
from langchain_community.retrievers import BM25Retriever

# Configure the sparse retriever (BM25) for medical data
bm25_medical_retriever = BM25Retriever.from_documents(medical_chunks)
bm25_medical_retriever.k = 2 # Retrieve 2 documents for sparse search

print("BM25 (sparse) medical retriever initialized successfully.")

# Configure the dense retriever (FAISS vector store) for medical data
dense_medical_retriever = medical_vector_store.as_retriever(search_kwargs={"k": 2}) # Retrieve 2 documents for dense search

print("FAISS (dense) medical retriever initialized successfully.")

print("Hybrid retrieval for medical data will be performed using a custom function.")

BM25 (sparse) medical retriever initialized successfully.
FAISS (dense) medical retriever initialized successfully.
Hybrid retrieval for medical data will be performed using a custom function.


In [35]:
import sys
!{sys.executable} -m pip install -qU rank_bm25

import numpy as np
from langchain_community.retrievers import BM25Retriever

# Configure the sparse retriever (BM25) for medical data
bm25_medical_retriever = BM25Retriever.from_documents(medical_chunks)
bm25_medical_retriever.k = 2 # Retrieve 2 documents for sparse search

print("BM25 (sparse) medical retriever initialized successfully.")

# Configure the dense retriever (FAISS vector store) for medical data
dense_medical_retriever = medical_vector_store.as_retriever(search_kwargs={"k": 2}) # Retrieve 2 documents for dense search

print("FAISS (dense) medical retriever initialized successfully.")

print("Hybrid retrieval for medical data will be performed using a custom function.")

BM25 (sparse) medical retriever initialized successfully.
FAISS (dense) medical retriever initialized successfully.
Hybrid retrieval for medical data will be performed using a custom function.


## Implement Multi-Query Generation for Medical Queries

### Subtask:
Develop a component that takes an initial medical user query and generates multiple alternative medical queries to broaden the search scope and improve recall within the medical domain, leveraging an LLM.


**Reasoning**:
I will define the `ChatPromptTemplate` for medical query generation, create the LLM chain, invoke it with a sample medical query, extract and parse the generated alternative queries, and then print both the original and alternative queries as per the instructions.



In [36]:
from langchain_core.prompts import ChatPromptTemplate

# 1. Define a ChatPromptTemplate for medical multi-query generation
medical_query_gen_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an AI medical assistant. Your task is to generate five different versions of the given medical user question to retrieve relevant medical documents from a knowledge base. By generating multiple perspectives on the user's question, your goal is to help the user overcome some of the limitations of distance-based similarity search in a medical context. Provide these alternative questions separated by newlines.",
        ),
        ("user", "{question}"),
    ]
)

print("Medical multi-query generation prompt template defined successfully.")

# 2. Create an LLM chain using LangChain Expression Language (LCEL)
medical_query_generation_chain = medical_query_gen_template | llm

# 3. Define a sample medical user query
original_medical_query = "What are the diagnostic uses of MRI?"

# 4. Invoke the chain to generate alternative queries
generated_medical_queries_response = medical_query_generation_chain.invoke({"question": original_medical_query})

# 5. Extract the text content from the response
generated_medical_queries_raw = generated_medical_queries_response.content

# 6. Parse the LLM's response to extract individual alternative medical queries
# Assuming the LLM provides queries separated by newlines
alternative_medical_queries = [q.strip() for q in generated_medical_queries_raw.split('\n') if q.strip()]

# 7. Print the original query and the generated alternative queries
print(f"\nOriginal Medical Query: {original_medical_query}")
print("\nGenerated Alternative Medical Queries:")
for i, query in enumerate(alternative_medical_queries):
    print(f"{i+1}. {query}")

Medical multi-query generation prompt template defined successfully.

Original Medical Query: What are the diagnostic uses of MRI?

Generated Alternative Medical Queries:
1. What medical conditions can be diagnosed using MRI technology?
2. How is MRI imaging utilized in medical diagnostics?
3. What are the applications of MRI scans in diagnosing health issues?
4. In what ways can MRI be helpful in identifying medical conditions?
5. How does MRI contribute to the diagnosis of various diseases and injuries?


## Implement Hybrid Search for Medical Context

### Subtask:
Combine sparse retrieval (e.g., BM25 for medical keywords) with dense retrieval (e.g., vector search for semantic understanding of medical concepts) to retrieve a diverse set of potentially relevant medical documents from the knowledge base, considering the original and multi-generated medical queries.


In [50]:
import numpy as np
from langchain_core.documents import Document

# 1. Define custom_hybrid_retriever function
def custom_hybrid_retriever(query, sparse_retriever, dense_retriever):
    # Retrieve documents from sparse retriever
    sparse_docs = sparse_retriever.invoke(query)

    # Retrieve documents from dense retriever
    dense_docs = dense_retriever.invoke(query)

    # Combine the retrieved documents
    combined_docs = sparse_docs + dense_docs

    # Remove duplicate documents based on page_content
    unique_docs_content = set()
    final_unique_docs = []
    for doc in combined_docs:
        if doc.page_content not in unique_docs_content:
            unique_docs_content.add(doc.page_content)
            final_unique_docs.append(doc)

    return final_unique_docs

print("Custom hybrid retriever function defined.")

# 2. Define all_medical_queries
# Ensure original_medical_query is defined from previous steps
# Ensure alternative_medical_queries is defined from previous steps

# Fallback if original_medical_query was not defined or multi-query generation failed
if 'original_medical_query' not in locals():
    original_medical_query = "What are the diagnostic uses of MRI?"

all_medical_queries = [original_medical_query]

# Check if alternative_medical_queries exists and is not empty
if 'alternative_medical_queries' in locals() and alternative_medical_queries:
    all_medical_queries.extend(alternative_medical_queries)

print(f"Total queries for retrieval: {len(all_medical_queries)}")

# 3. Initialize an empty list called all_retrieved_medical_docs
all_retrieved_medical_docs = []

# 4. Iterate through each query in all_medical_queries
# 5. For each query, call the custom_hybrid_retriever function
# 6. Extend the all_retrieved_medical_docs list
print("Performing hybrid retrieval for all medical queries...")
for query in all_medical_queries:
    retrieved_docs_for_query = custom_hybrid_retriever(query, bm25_medical_retriever, dense_medical_retriever)
    all_retrieved_medical_docs.extend(retrieved_docs_for_query)

# 7. Create a new list final_retrieved_medical_docs containing only unique documents
unique_docs_content_final = set()
final_retrieved_medical_docs = []
for doc in all_retrieved_medical_docs:
    if doc.page_content not in unique_docs_content_final:
        unique_docs_content_final.add(doc.page_content)
        final_retrieved_medical_docs.append(doc)

# 8. Print the number of unique documents retrieved and an example
print(f"Retrieved {len(final_retrieved_medical_docs)} unique documents using hybrid search for medical queries.")
if final_retrieved_medical_docs:
    print("Example of a retrieved medical document (first 200 characters):")
    print(f"{final_retrieved_medical_docs[0].page_content[:200]}...")
else:
    print("No documents retrieved for medical queries.")

Custom hybrid retriever function defined.
Total queries for retrieval: 6
Performing hybrid retrieval for all medical queries...
Retrieved 10 unique documents using hybrid search for medical queries.
Example of a retrieved medical document (first 200 characters):
monitoring of cardiovascular risk factors.
36. Epidemiology: Medical knowledge entry describing diseases,
    mechanisms, symptoms, diagnostics, and treatments. Example concept:
    Hypertension invol...


In [52]:
import numpy as np
from langchain_core.documents import Document

# 1. Define custom_hybrid_retriever function
def custom_hybrid_retriever(query, sparse_retriever, dense_retriever):
    # Retrieve documents from sparse retriever
    sparse_docs = sparse_retriever.invoke(query)

    # Retrieve documents from dense retriever
    dense_docs = dense_retriever.invoke(query)

    # Combine the retrieved documents
    combined_docs = sparse_docs + dense_docs

    # Remove duplicate documents based on page_content
    unique_docs_content = set()
    final_unique_docs = []
    for doc in combined_docs:
        if doc.page_content not in unique_docs_content:
            unique_docs_content.add(doc.page_content)
            final_unique_docs.append(doc)

    return final_unique_docs

print("Custom hybrid retriever function defined.")

# 2. Define all_medical_queries
# Ensure original_medical_query is defined from previous steps
# Ensure alternative_medical_queries is defined from previous steps

# Fallback if original_medical_query was not defined or multi-query generation failed
if 'original_medical_query' not in locals():
    original_medical_query = "What are the diagnostic uses of MRI?"

all_medical_queries = [original_medical_query]

# Check if alternative_medical_queries exists and is not empty
if 'alternative_medical_queries' in locals() and alternative_medical_queries:
    all_medical_queries.extend(alternative_medical_queries)

print(f"Total queries for retrieval: {len(all_medical_queries)}")

# 3. Initialize an empty list called all_retrieved_medical_docs
all_retrieved_medical_docs = []

# 4. Iterate through each query in all_medical_queries
# 5. For each query, call the custom_hybrid_retriever function
# 6. Extend the all_retrieved_medical_docs list
print("Performing hybrid retrieval for all medical queries...")
for query in all_medical_queries:
    retrieved_docs_for_query = custom_hybrid_retriever(query, bm25_medical_retriever, dense_medical_retriever)
    all_retrieved_medical_docs.extend(retrieved_docs_for_query)

# 7. Create a new list final_retrieved_medical_docs containing only unique documents
unique_docs_content_final = set()
final_retrieved_medical_docs = []
for doc in all_retrieved_medical_docs:
    if doc.page_content not in unique_docs_content_final:
        unique_docs_content_final.add(doc.page_content)
        final_retrieved_medical_docs.append(doc)

# 8. Print the number of unique documents retrieved and an example
print(f"Retrieved {len(final_retrieved_medical_docs)} unique documents using hybrid search for medical queries.")
if final_retrieved_medical_docs:
    print("Example of a retrieved medical document (first 200 characters):")
    print(f"{final_retrieved_medical_docs[0].page_content[:200]}...")
else:
    print("No documents retrieved for medical queries.")

Custom hybrid retriever function defined.
Total queries for retrieval: 6
Performing hybrid retrieval for all medical queries...
Retrieved 10 unique documents using hybrid search for medical queries.
Example of a retrieved medical document (first 200 characters):
monitoring of cardiovascular risk factors.
36. Epidemiology: Medical knowledge entry describing diseases,
    mechanisms, symptoms, diagnostics, and treatments. Example concept:
    Hypertension invol...


## Implement Reranking Models for Medical Relevance

### Subtask:
Apply a reranking model to the medical documents retrieved by the hybrid search to re-score and reorder them for optimal relevance to the medical query, prioritizing highly pertinent clinical or research information.


In [53]:
from sentence_transformers import CrossEncoder

# 1. Load a pre-trained cross-encoder model for medical reranking
medical_reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

print("Medical reranking model 'cross-encoder/ms-marco-MiniLM-L-6-v2' loaded successfully.")

# 3. Prepare query-document pairs for reranking
# The 'final_retrieved_medical_docs' list contains Document objects.
# The 'original_medical_query' holds the user's initial question.
medical_query_document_pairs = []
for doc in final_retrieved_medical_docs:
    medical_query_document_pairs.append([original_medical_query, doc.page_content])

# 4. Use the loaded reranker model to predict scores
# The reranker.predict method takes a list of [query, document] pairs
# and returns a list of scores.
medical_rerank_scores = medical_reranker.predict(medical_query_document_pairs)

# 5. Combine the original documents with their predicted reranking scores
# Create a list of tuples: (score, document)
medical_reranked_documents_with_scores = []
for i, score in enumerate(medical_rerank_scores):
    medical_reranked_documents_with_scores.append((score, final_retrieved_medical_docs[i]))

# 6. Sort the documents in descending order based on their reranking scores
sorted_medical_reranked_documents = sorted(medical_reranked_documents_with_scores, key=lambda x: x[0], reverse=True)

# 7. Print the top reranked documents
print("\n--- Reranked Medical Documents (Top 3) ---")
for i, (score, doc) in enumerate(sorted_medical_reranked_documents[:3]):
    print(f"Rank {i+1} (Score: {score:.4f}):")
    print(f"Content: {doc.page_content[:200]}...") # Print first 200 chars
    print("----------------------------------")

print("Medical reranking process completed and top documents displayed.")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Medical reranking model 'cross-encoder/ms-marco-MiniLM-L-6-v2' loaded successfully.

--- Reranked Medical Documents (Top 3) ---
Rank 1 (Score: -6.4068):
Content: that increases risk of stroke, myocardial infarction, kidney
    disease, and heart failure. Management may include lifestyle
    modification, antihypertensive medications such as ACE inhibitors,
   ...
----------------------------------
Rank 2 (Score: -6.6629):
Content: that increases risk of stroke, myocardial infarction, kidney
     disease, and heart failure. Management may include lifestyle
     modification, antihypertensive medications such as ACE inhibitors,
 ...
----------------------------------
Rank 3 (Score: -6.7769):
Content: 942. Human Anatomy: Medical knowledge entry describing diseases,
     mechanisms, symptoms, diagnostics, and treatments. Example concept:
     Hypertension involves chronically elevated arterial blood...
----------------------------------
Medical reranking process completed and top documents

In [56]:
from langchain_core.prompts import ChatPromptTemplate

# 1. Define a ChatPromptTemplate for the LLM to generate a medical answer
medical_answer_gen_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an AI medical assistant. Answer the user's medical question based ONLY on the provided context about medical imaging techniques. "
            "If you cannot find the answer in the context, clearly state that you cannot answer. "
            "Do not make up any medical information. Keep your answer concise and factual."
            "Context: {context}",
        ),
        ("user", "{question}"),
    ]
)

# 2. Create an LLM chain by combining the prompt template and the initialized llm
medical_response_generation_chain = medical_answer_gen_template | llm

print("Medical response generation prompt template defined and LLM chain created successfully.")

# 3. Format the sorted_medical_reranked_documents into a single string for context
context_for_medical_llm = "\n\n---\n\n".join([doc.page_content for score, doc in sorted_medical_reranked_documents])

print("Invoking LLM for medical answer generation...")
# 4. Invoke the medical_response_generation_chain with the formatted context and the original medical query
medical_response = medical_response_generation_chain.invoke(
    {"context": context_for_medical_llm, "question": original_medical_query}
)

# 5. Extract the generated medical answer.
generated_medical_answer = medical_response.content

# 6. Print the generated medical answer.
print(f"\n--- Generated Medical Answer ---\n{generated_medical_answer}")
print("\nMedical response generation completed.")

Medical response generation prompt template defined and LLM chain created successfully.
Invoking LLM for medical answer generation...

--- Generated Medical Answer ---
I cannot answer that based on the provided context.

Medical response generation completed.
